# Kyrgyz morphology: QLoRA fine-tuning

Teaches an open model Kyrgyz noun inflection, then measures the gain on held-out words.

Pipeline: split stems into train/test, generate training pairs from the train stems with the
verified rule engine, score the base model on the **test** stems, fine-tune with QLoRA, score again.
The test stems are never seen in training, so any gain is real generalization of the rules.

**Before running:** Settings -> Accelerator -> GPU (T4 or P100), and Internet -> On.


## 1. Install

In [ ]:
!pip install -q -U "transformers==4.44.2" "peft==0.12.0" "trl==0.9.6" "bitsandbytes==0.43.3" "accelerate==0.33.0" "datasets==2.21.0"
# If imports fail on a newer Kaggle image, remove the version pins and re-run.

## 2. Get the verified rules and stems

In [ ]:
import sys, subprocess, os
if not os.path.exists("kyrgyz-llm-benchmark"):
    subprocess.run(["git", "clone", "-q", "https://github.com/yryssqq/kyrgyz-llm-benchmark"], check=True)
sys.path.insert(0, "kyrgyz-llm-benchmark/src")

from kyrgyz_eval import morphology as m

def read_stems(path):
    out = []
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        if "|" in line:
            s, irr = line.split("|", 1)
            out.append((s.strip(), irr.strip()))
        else:
            out.append((line, None))
    return out

from pathlib import Path
STEMS = read_stems("kyrgyz-llm-benchmark/data/stems.example.txt")
print(len(STEMS), "stems")

## 3. Config

`Qwen2.5-3B` fits comfortably on a free T4/P100 and trains in minutes. Switch to `Qwen2.5-7B-Instruct`
for a stronger base once the pipeline works. Expand `stems.example.txt` (30 -> 300 words) for a real run.


In [ ]:
MODEL_ID   = "Qwen/Qwen2.5-3B-Instruct"
OUTPUT_DIR = "/kaggle/working/kyrgyz-morph-lora"
FEATURES   = ("plural",) + m.CASES + m.POSSESSIVES
MAX_NEW    = 8
EPOCHS     = 3
LR         = 2e-4
BATCH      = 8
SEED       = 42

## 4. Train/test split and data generation

In [ ]:
import random
random.seed(SEED)

stems = STEMS[:]
random.shuffle(stems)
n_test = max(1, len(stems) // 5)
test_stems, train_stems = stems[:n_test], stems[n_test:]

def instruction(stem, feature):
    return f"«{stem}» сөзүнүн {m.FEATURE_LABELS_KY[feature]} түрүн жазыңыз."

def make_pairs(stem_list):
    pairs = []
    for stem, irr in stem_list:
        for feat in FEATURES:
            pairs.append({"instruction": instruction(stem, feat),
                          "output": m.inflect(stem, feat, irregular=irr)})
    return pairs

train_pairs = make_pairs(train_stems)
test_pairs  = make_pairs(test_stems)
print(len(train_pairs), "train pairs |", len(test_pairs), "test pairs")
print("example:", train_pairs[0])

## 5. Load the base model in 4-bit

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map="auto")
model.config.use_cache = False

## 6. Evaluation (exact match on held-out stems)

In [ ]:
@torch.no_grad()
def evaluate(model, pairs):
    model.eval()
    correct = 0
    for p in pairs:
        msgs = [{"role": "user", "content": p["instruction"]}]
        prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        ids = tokenizer(prompt, return_tensors="pt").to(model.device)
        out = model.generate(**ids, max_new_tokens=MAX_NEW, do_sample=False,
                             pad_token_id=tokenizer.pad_token_id)
        text = tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
        if text.strip().split()[:1] == [p["output"]] or text.strip() == p["output"]:
            correct += 1
    return correct / len(pairs)

before = evaluate(model, test_pairs)
print(f"base model, held-out accuracy: {before:.1%}")

## 7. Fine-tune with QLoRA

In [ ]:
from datasets import Dataset
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

def to_text(p):
    msgs = [{"role": "user", "content": p["instruction"]},
            {"role": "assistant", "content": p["output"]}]
    return {"text": tokenizer.apply_chat_template(msgs, tokenize=False)}

train_ds = Dataset.from_list([to_text(p) for p in train_pairs])

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])

cfg = SFTConfig(output_dir=OUTPUT_DIR, num_train_epochs=EPOCHS, per_device_train_batch_size=BATCH,
                gradient_accumulation_steps=2, learning_rate=LR, warmup_ratio=0.05,
                logging_steps=10, save_strategy="no", bf16=True, max_seq_length=128,
                dataset_text_field="text", report_to="none", seed=SEED)

trainer = SFTTrainer(model=model, args=cfg, train_dataset=train_ds, peft_config=lora)
trainer.train()

## 8. Re-evaluate and compare

In [ ]:
after = evaluate(trainer.model, test_pairs)
print(f"before: {before:.1%}   after: {after:.1%}   delta: {after-before:+.1%}")

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(4.5, 4))
bars = ax.bar(["base", "fine-tuned"], [before, after], color=["#c44", "#2a8"])
ax.set_ylim(0, 1); ax.set_ylabel("held-out inflection accuracy")
ax.set_title("Kyrgyz morphology: QLoRA")
for b, v in zip(bars, [before, after]):
    ax.text(b.get_x()+b.get_width()/2, v+0.02, f"{v:.0%}", ha="center")
plt.tight_layout(); plt.savefig("/kaggle/working/before_after.png", dpi=150); plt.show()

## 9. Save the adapter

In [ ]:
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("saved LoRA adapter to", OUTPUT_DIR)